In [0]:
import pandas as pd

df = pd.read_csv("/Workspace/Users/kushal.vadukul2006@gmail.com/Multi-Class-Credit-Card-Risk-Model/UCI_Credit_Card.csv")

df

In [0]:

df1 = df.copy()

def risk(x):
    if x <= 0:
        return 0
    elif x == 1:
        return 1
    elif x == 2:
        return 2
    else:
        return 3
        

df1['Credit_Risk'] = df['PAY_0'].apply(risk)

df1



In [0]:
%pip install scikit-optimize
%pip install xgboost
%pip install pytorch-tabular
%pip install optuna

In [0]:
from sklearn.model_selection import train_test_split

df1['maxedout_percent'] = (df1['BILL_AMT1']) / (df1['LIMIT_BAL'] +1)
df1['paid_percent'] = df1['PAY_AMT1'] / (df1['BILL_AMT1'].abs() + 1)
df1['pay_increase'] = (df1['PAY_AMT1'] - df1['PAY_AMT6']) / (df1['PAY_AMT6'] + 1)

features = ['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','EDUCATION']

target = 'Credit_Risk'


X_train, X_test, y_train, y_test = train_test_split(df1[features], df1[target], test_size=0.2, random_state=42)

train_df1 = X_train.copy()
train_df1[target] = y_train
education_risk = train_df1.groupby('EDUCATION')[target].mean()
df1['education_risk'] = df1['EDUCATION'].map(education_risk)



features = ['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','EDUCATION','education_risk']

X_train, X_test, y_train, y_test = train_test_split(df1[features], df1[target], test_size=0.2, random_state=42)

In [0]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, f1_score

Model = xgb.XGBClassifier(objective='multi:softmax', num_class=4, random_state=42)

Model.fit(X_train, y_train)

y_prediction = Model.predict(X_test)

print(classification_report(y_test, y_prediction))
print(confusion_matrix(y_test, y_prediction))
macro_f1 = f1_score(y_test, y_prediction, average='macro')
print(f"Macro F1-Score: {macro_f1:.4f}")






In [0]:
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, ModelConfig, TrainerConfig, OptimizerConfig
from pytorch_tabular.models import FTTransformerConfig
from sklearn.metrics import f1_score
from pytorch_tabular.utils import get_balanced_sampler


train_df2 = X_train.copy()
train_df2['Credit_Risk'] = y_train
test_df2 = X_test.copy()
test_df2['Credit_Risk'] = y_test


data = DataConfig(
    target=['Credit_Risk'],
    continuous_cols=['PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6','education_risk'],
    categorical_cols=['EDUCATION'],
    
)


trainer = TrainerConfig(batch_size=64, max_epochs=10, accelerator="auto", early_stopping=None)

optimizer = OptimizerConfig()


model = FTTransformerConfig(
    task="classification",
    input_embed_dim=32,
    num_attn_blocks=3,
    num_heads=4,
    learning_rate=1e-3,

)


tabular_model = TabularModel(
    data_config=data,
    model_config=model,
    optimizer_config=optimizer,
    trainer_config=trainer
)

train_sampler = get_balanced_sampler(train_df2['Credit_Risk'])

tabular_model.fit(train=train_df2, validation=test_df2, train_sampler=train_sampler)

result = tabular_model.evaluate(test=test_df2)
print(result)


preds_df = tabular_model.predict(test_df2)

y_pred = preds_df['Credit_Risk_prediction']
true_macro_f1 = f1_score(test_df2['Credit_Risk'], y_pred, average='macro')

print(f"True FT-Transformer Macro F1 Score: {true_macro_f1:.4f}")


In [0]:


preds_df